# LangChain Core Concepts + Tiny RAG — Day 16

Covers `daily-plans/month-1/day-016.md`: **LCEL, document loaders, text splitters,
and a tiny RAG pipeline over 1 PDF** — using the real Apple 10-K you already
downloaded on Day 15, not a toy file.

**Why LangChain at all, at this point in the curriculum:** you already know how to
do every individual piece by hand —
[Day 13](13Embeddings.ipynb) embeddings + cosine similarity,
[Day 14](../month-1/14ChromaDb.py) storing vectors in Chroma,
[Day 15](15Pypdf-Plumber.ipynb) pulling text out of a PDF. LangChain doesn't
introduce new *concepts* here — it gives you a **consistent interface** so those
pieces snap together as a pipeline instead of you hand-wiring every step every
time. That's the whole value proposition, no more, no less.

In [1]:
from dotenv import load_dotenv
load_dotenv()
print("Environment loaded.")

Environment loaded.


## 1. LCEL — the `|` pipe operator

LCEL (**L**ang**C**hain **E**xpression **L**anguage) is just Python's `|` operator
overloaded to mean "feed the output of the left side into the right side." That's
the entire concept — everything else in this notebook is just longer pipes built
from the same idea.

Three pieces, chained:
- **Prompt template** — turns your variables into a formatted message
- **Model** — sends that message to the LLM, gets a response object back
- **Output parser** — pulls the plain string out of that response object

Try the smallest possible pipe first, no PDFs yet.

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_template("Answer in one short sentence: {question}")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

chain = prompt | llm | parser   # <-- this is the entire LCEL concept

result = chain.invoke({"question": "What is a 10-K filing?"})
print(result)

A 10-K filing is an annual report required by the SEC that provides a comprehensive overview of a company's financial performance, including audited financial statements and detailed information about its operations.


Read `prompt | llm | parser` left to right: your input dict goes into `prompt`,
`prompt`'s formatted message goes into `llm`, `llm`'s response object goes into
`parser`, and `parser`'s plain string is what `.invoke()` finally returns. No loop,
no manual "call this, take the output, feed it into that" — the `|` does it.

## 2. Document loaders

Real-world text doesn't arrive as a clean Python string — it's in PDFs, web pages,
CSVs, Notion pages, etc. A **document loader** is LangChain's answer: no matter the
source, it hands you back a list of `Document` objects, each with the same shape
(`.page_content` = the text, `.metadata` = where it came from).

This is doing the same job as `pypdf`/`pdfplumber` did by hand on Day 15 — just
wrapped in an interface every other LangChain piece expects.

In [3]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "../tracker/projects/1-sec-10k-analyzer/AAPL_10K_2025-10-31.pdf"

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

print("Pages loaded:", len(docs))
print("\nType of each item:", type(docs[0]))
print("\nFirst page metadata:", docs[0].metadata)
print("\nFirst page content (first 300 chars):\n", docs[0].page_content[:300])

/var/folders/h9/_gsfr5b96534t5201zpwxl180000gn/T/ipykernel_20270/4222216949.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Pages loaded: 111

Type of each item: <class 'langchain_core.documents.base.Document'>

First page metadata: {'producer': 'WeasyPrint 69.0', 'creator': 'PyPDF', 'creationdate': '', 'title': 'aapl-20250927', 'source': '../tracker/projects/1-sec-10k-analyzer/AAPL_10K_2025-10-31.pdf', 'total_pages': 111, 'page': 0, 'page_label': '1'}

First page content (first 300 chars):
 UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☒ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended September 27, 2025
or
☐TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCH


One `Document` per page — 111 of them, matching what `pypdf` reported directly on
Day 15. Same underlying extraction, same 111-page count, just returned as
`Document` objects instead of raw strings.

## 3. Text splitters

Straight from Day 13's finding: a long, multi-topic chunk of text embeds as a
diluted blend (remember the 0.48 "topic buried in noise" score). A **text
splitter** breaks big documents into smaller, more topically-focused chunks before
embedding — this is the actual mechanism behind the chunking strategy Section 6 of
`13Embeddings.ipynb` described in theory.

Two settings that matter:
- `chunk_size` — max characters per chunk
- `chunk_overlap` — characters shared between consecutive chunks, so a sentence
  that straddles a chunk boundary doesn't get cut in half and lose meaning in both
  pieces

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(docs)

print(f"{len(docs)} pages -> {len(chunks)} chunks")
print("\nExample chunk:\n", chunks[10].page_content[:400])
print("\nIts metadata (still knows which page it came from):", chunks[10].metadata)

111 pages -> 294 chunks

Example chunk:
 business and results of operations are forward-looking statements. Forward-looking statements can also be identified
by words such as “future,” “anticipates,” “believes,” “estimates,” “expects,” “intends,” “plans,” “predicts,” “will,” “would,”
“could,” “can,” “may,” and similar terms. Forward-looking statements are not guarantees of future performance and the
Company’s actual results may differ si

Its metadata (still knows which page it came from): {'producer': 'WeasyPrint 69.0', 'creator': 'PyPDF', 'creationdate': '', 'title': 'aapl-20250927', 'source': '../tracker/projects/1-sec-10k-analyzer/AAPL_10K_2025-10-31.pdf', 'total_pages': 111, 'page': 4, 'page_label': '5'}


Notice the chunk still carries `metadata` (which page it came from) — that survived
the split. That matters later: when a chunk gets retrieved as an answer, you can
still tell the user *which page* of the filing it came from, not just the text.

## 4. Embeddings + vector store (Days 13-14, through LangChain's interface)

Same `text-embedding-3-small` model from Day 13, same Chroma database from Day 14
— just called through LangChain's `OpenAIEmbeddings` and `Chroma` wrappers instead
of the raw OpenAI/Chroma clients, so they plug directly into an LCEL chain.

In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# using a subset of chunks here so this cell runs in seconds, not minutes --
# swap in `chunks` (all ~294 of them) for the full document
vectorstore = Chroma.from_documents(chunks[:60], embeddings, collection_name="aapl_10k_day16")

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Vector store ready. Retriever will return the top", 3, "matching chunks per query.")

Vector store ready. Retriever will return the top 3 matching chunks per query.


## 5. The full RAG chain — everything piped together

This is the actual architecture described in Section 6 of `13Embeddings.ipynb`,
built for real: retrieve the most relevant chunks for a question, hand them to the
LLM as context, get an answer grounded in the real filing.

In [7]:
from langchain_core.runnables import RunnablePassthrough

def format_docs(retrieved_docs):
    return "\n\n".join(d.page_content for d in retrieved_docs)

rag_prompt = ChatPromptTemplate.from_template(
    "Answer the question using ONLY the context below. "
    "If the context doesn't contain the answer, say so.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | parser
)

print("RAG chain built.")

RAG chain built.


Read that dict-then-pipe carefully, since it looks different from the simple
3-step pipe in Section 1:

```
{"context": retriever | format_docs, "question": RunnablePassthrough()}
```

This dict runs **two things in parallel** on the same input question:
- `"context"`: sends the question into `retriever` (finds the top-3 matching
  chunks), then `format_docs` (joins them into one string)
- `"question"`: `RunnablePassthrough()` just means "pass the original question
  through unchanged"

Both results land in a dict with those two keys, which is exactly what
`rag_prompt`'s `{context}` and `{question}` placeholders expect. Then it's the
same pattern as Section 1: `prompt | llm | parser`.

In [8]:
result = rag_chain.invoke("What products and services does Apple sell?")
print(result)

Apple sells a variety of products and services, including:

1. **Advertising Services** - Third-party licensing arrangements and its own advertising platforms.
2. **AppleCare** - Fee-based service and support products providing technical support, repair services, and coverage for accidental damage or theft.
3. **Cloud Services** - Services that store and keep customers' content up-to-date across multiple devices.
4. **Digital Content** - Platforms like the App Store for discovering and downloading applications, books, music, video, games, and podcasts.
5. **Mac** - Personal computers including MacBook Air, MacBook Pro, iMac, Mac mini, Mac Studio, and Mac Pro.
6. **iPad** - Multipurpose tablets including iPad Pro, iPad Air, iPad, and iPad mini.
7. **Wearables, Home and Accessories** - Smartwatches (Apple Watch Series 11, Apple Watch SE 3), wireless headphones, and spatial computers.
8. **Subscription-based Digital Content Services** - Apple Arcade, Apple Fitness+, Apple Music, Apple New

In [9]:
result = rag_chain.invoke("What was Apple's total net sales for fiscal year 2025?")
print(result)

The context does not contain the answer to the question about Apple's total net sales for fiscal year 2025.


## 6. Play with it

- Ask your own questions about Apple's business — try something the first 60 chunks
  might *not* cover, and see if the chain honestly says it doesn't know (it should,
  per the prompt's instruction) rather than making something up.
- Swap `chunks[:60]` for the full `chunks` list in Section 4 to search the entire
  111-page filing instead of a subset.
- Try `search_kwargs={"k": 5}` instead of `3` — more retrieved chunks, more context,
  sometimes a better answer, always a bigger/slower prompt.

In [10]:
# your playground -- ask your own question


## 7. What you just built

This *is* Project 1's core engine, at small scale:

```
PDF  →  PyPDFLoader  →  RecursiveCharacterTextSplitter  →  OpenAIEmbeddings  →  Chroma
                                                                                    │
User question  ─────────────────────────────────────────────────────────►  retriever
                                                                                    │
                                                                        top-k chunks
                                                                                    │
                                                                    rag_prompt + LLM
                                                                                    │
                                                                          grounded answer
```

The production version (Project 1 proper) swaps Chroma for **Pinecone** (Day 14 —
hosted, scales past a local folder), adds **multi-document** support (compare
across AAPL/MSFT/JPM), and wraps this chain in a **FastAPI + Streamlit** app instead
of a notebook. The chain logic itself — load, split, embed, retrieve, prompt — stays
exactly the same.